# Distributed Computing with Ray on GKE from Kubeflow Notebooks v2

Welcome! This guide is designed for **data scientists, ML researchers, and engineers** who want to run distributed Python workloads (parallel data processing, multi-node model training, or hyperparameter sweeps) on **Google Kubernetes Engine (GKE)** without needing to be Kubernetes experts.

---

### What is this and how does it work?

* **Your Kubeflow Notebook (Workspace)**: Think of this as your interactive "laptop" in the cloud. It runs on minimal, inexpensive resources (e.g. 0.1 CPU, 128 MB RAM) so it can stay active all day without incurring high cloud costs.
* **Ray**: A popular open-source Python framework that lets you scale any Python function or class across a cluster by simply adding the `@ray.remote` decorator.
* **KubeRay Operator**: An automated controller running in your GKE cluster that creates and manages Ray clusters on-demand.
* **KubeRay Python Client SDK**: A pure-Python library (`python-client`) that lets you create, inspect, and delete Ray clusters directly from your notebook code—no `kubectl` commands or Kubernetes YAML files required!

```
  ┌─────────────────────────────────────────────────────────────────┐
  │       Kubeflow Notebook v2 Pod (Tiny CPU: 0.1 CPU, 128Mi)       │
  │     Interactive Development, Notebook UI, Pure Python Code      │
  └───────────────┬─────────────────────────────────┬───────────────┘
                  │ 1. Launch / Delete Cluster      │ 2. Submit Tasks / Jobs
                  ▼ (KubeRay Python Client SDK)     ▼ (Ray Client :10001 / Jobs API :8265)
  ┌───────────────────────────────┐ ┌───────────────────────────────────────────────┐
  │     KubeRay Operator          │ │           RayCluster (ray.io/v1)              │
  │  (Automated Cluster Manager)  │ │  ┌─────────────────┐   ┌───────────────────┐  │
  └───────────────────────────────┘ │  │  Ray Head Pod   │   │  Ray Worker Pods  │  │
                                    │  │ (Dashboard, GCS │──▶│  (Distributed     │  │
                                    │  │  Client Server) │   │   Task Execution) │  │
                                    │  └─────────────────┘   └───────────────────┘  │
                                    └───────────────────────────────────────────────┘
```

### Jargon Buster (Key Terms Explained)
* **Namespace**: Your team's private project folder in the cluster (e.g., `kubeflow-user-example-com`). Everything created in this notebook stays within your namespace.
* **Head Node**: The "manager" of the Ray cluster. It hosts the Ray Dashboard web UI, coordinates distributed tasks, and serves client connections.
* **Worker Nodes**: The "workers" that execute your computational tasks in parallel across CPUs or GPUs.
* **Ray Client (`ray://...:10001`)**: An interactive connection that lets code running in this notebook transparently execute on the remote Ray workers.
* **Ray Jobs API (`http://...:8265`)**: An asynchronous job runner. You submit a Python script, and it continues executing on the cluster even if you close this notebook or shut down your laptop.

## Step 0: Install Dependencies & Verify Environment

To manage the Ray cluster and run distributed code, we need two Python packages:
1. **`ray[default,client]`**: The core Ray SDK used to connect to the cluster (`ray.init`) and submit batch jobs (`JobSubmissionClient`).
2. **`python-client` (KubeRay Python SDK)**: The official client from the [KubeRay project](https://github.com/ray-project/kuberay/tree/master/clients/python-client) that provides high-level Python APIs (`RayClusterApi`, `ClusterBuilder`) to manage Ray clusters without raw Kubernetes calls.

Both can be installed directly with `pip`:
```bash
pip install "ray[default,client]==2.58.0"
pip install "git+https://github.com/ray-project/kuberay.git#subdirectory=clients/python-client"
# Or from local source: pip install /path/to/kuberay/clients/python-client
```

> 💡 **Kernel Restart Note:** If packages are installed for the first time in your workspace, restart the kernel (**Kernel ➔ Restart Kernel...**) so Python loads the newly installed C extensions (`_raylet.so`) cleanly into memory.

In [ ]:
import os
import sys
import subprocess

# 1. Install Ray and the KubeRay Python Client SDK if not already present
RAY_VERSION = "2.58.0"
KUBERAY_CLIENT_REPO = "git+https://github.com/ray-project/kuberay.git#subdirectory=clients/python-client"

packages_to_install = []

try:
    import ray
    assert ray.__version__ == RAY_VERSION
except (ImportError, AssertionError):
    packages_to_install.append(f"ray[default,client]=={RAY_VERSION}")

try:
    import python_client
except ImportError:
    packages_to_install.append(KUBERAY_CLIENT_REPO)

if packages_to_install:
    print(f"📦 Installing required packages: {packages_to_install}...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet"] + packages_to_install)
    print("✅ Installation complete.")
    print("💡 Tip: After installing Ray for the first time, restart the notebook kernel (Kernel ➔ Restart Kernel...) to load newly compiled C extensions cleanly.")

import ray
from python_client import kuberay_cluster_api
from python_client.utils import kuberay_cluster_builder

# 2. Detect the current Kubeflow Workspace tenant namespace automatically
# Kubeflow mounts the tenant namespace into /var/run/secrets/kubernetes.io/serviceaccount/namespace
SA_NAMESPACE_FILE = "/var/run/secrets/kubernetes.io/serviceaccount/namespace"
if os.path.exists(SA_NAMESPACE_FILE):
    with open(SA_NAMESPACE_FILE, "r") as f:
        TENANT_NAMESPACE = f.read().strip()
else:
    TENANT_NAMESPACE = "default"

print(f"🐍 Python version   : {sys.version.split()[0]}")
print(f"⚡ Ray SDK version  : {ray.__version__}")
print(f"📁 Tenant Namespace : {TENANT_NAMESPACE}")

## Step 1: Define Your Ray Cluster Using the KubeRay SDK

Instead of manually editing complex Kubernetes YAML manifests, we use the KubeRay Python SDK:
- **`kuberay_cluster_builder.ClusterBuilder`**: A fluent Python builder to configure your cluster (head node, worker group, CPU, memory, replica count).
- **`kuberay_cluster_api.RayClusterApi`**: The Python client to create, list, inspect status, and delete Ray clusters.

### Key Configuration Parameters:
- **`cluster_name`**: Name of the Ray cluster (e.g., `raycluster-sample`).
- **`num_workers`**: Number of worker machines to provision.
- **`worker_cpu` / `worker_memory`**: CPU cores and RAM allocated to each worker pod.
- **`image`**: Docker container image for Ray. **Important:** The Python minor version in the container (`py311`) must match your notebook environment (`Python 3.11`) for Ray Client serialization to work smoothly.
- **Istio sidecar injection**: Kubeflow tenant namespaces have Istio enabled. Ray worker nodes communicate directly over dynamic gRPC ports, so we automatically disable Istio sidecar injection on the Ray pods (`sidecar.istio.io/inject: "false"`) to ensure uninhibited high-performance communication.
- **GKE Node Scheduling**: If your GKE cluster uses gVisor Sandbox nodes for notebooks (Pause & Resume), Ray pods automatically schedule onto standard (non-gVisor) node pools, as Ray requires native Linux POSIX shared memory (`/dev/shm`) for its Plasma object store.

In [ ]:
from typing import Optional, Tuple
from python_client import kuberay_cluster_api
from python_client.utils import kuberay_cluster_builder

def build_ray_cluster_spec(
    cluster_name: str = "raycluster-sample",
    tenant_namespace: str = TENANT_NAMESPACE,
    ray_version: str = RAY_VERSION,
    num_workers: int = 1,
    min_workers: int = 1,
    max_workers: int = 2,
    head_cpu_request: str = "250m",
    head_cpu_limit: str = "500m",
    head_memory_request: str = "1Gi",
    head_memory_limit: str = "2Gi",
    worker_cpu_request: str = "500m",
    worker_cpu_limit: str = "1",
    worker_memory_request: str = "1Gi",
    worker_memory_limit: str = "2Gi",
    image: Optional[str] = None,
) -> dict:
    """Build a RayCluster specification dictionary using KubeRay's ClusterBuilder."""
    if image is None:
        py_tag = f"py{sys.version_info.major}{sys.version_info.minor}"
        image = f"rayproject/ray:{ray_version}-{py_tag}"

    builder = kuberay_cluster_builder.ClusterBuilder()
    cluster = (
        builder.build_meta(
            name=cluster_name,
            k8s_namespace=tenant_namespace,
            labels={
                "app.kubernetes.io/name": cluster_name,
                "app.kubernetes.io/part-of": "kubeflow-notebooks-ray",
            },
            ray_version=ray_version,
        )
        .build_head(
            ray_image=image,
            service_type="ClusterIP",
            cpu_requests=head_cpu_request,
            memory_requests=head_memory_request,
            cpu_limits=head_cpu_limit,
            memory_limits=head_memory_limit,
            ray_start_params={
                "dashboard-host": "0.0.0.0",
                "num-cpus": "0",  # Keep head node dedicated to cluster coordination
            },
        )
        .build_worker(
            group_name="workers",
            ray_image=image,
            cpu_requests=worker_cpu_request,
            memory_requests=worker_memory_request,
            cpu_limits=worker_cpu_limit,
            memory_limits=worker_memory_limit,
            replicas=num_workers,
            min_replicas=min_workers,
            max_replicas=max_workers,
        )
        .get_cluster()
    )

    # Disable Istio sidecar injection on Ray pods so Ray internal gRPC communication
    # on dynamic ephemeral ports is not blocked by Istio mTLS AuthorizationPolicies
    cluster["spec"]["headGroupSpec"]["template"].setdefault("metadata", {})["labels"] = {
        "ray.io/node-type": "head",
        "sidecar.istio.io/inject": "false",
    }
    for wg in cluster["spec"]["workerGroupSpecs"]:
        wg["template"].setdefault("metadata", {})["labels"] = {
            "ray.io/node-type": "worker",
            "sidecar.istio.io/inject": "false",
        }

    return cluster


def launch_ray_cluster(
    cluster_name: str = "raycluster-sample",
    tenant_namespace: str = TENANT_NAMESPACE,
    num_workers: int = 1,
    timeout_seconds: int = 300,
) -> Tuple[str, str]:
    """Launch a Ray cluster using KubeRay SDK and wait for it to be ready.

    Returns:
        (ray_client_uri, ray_dashboard_uri)
    """
    cluster_api = kuberay_cluster_api.RayClusterApi()
    cluster_spec = build_ray_cluster_spec(
        cluster_name=cluster_name,
        tenant_namespace=tenant_namespace,
        num_workers=num_workers,
    )

    print(f"🚀 Submitting RayCluster '{cluster_name}' in tenant namespace '{tenant_namespace}' (workers={num_workers})...")
    created = cluster_api.create_ray_cluster(body=cluster_spec, k8s_namespace=tenant_namespace)
    if created is None:
        print(f"ℹ️ RayCluster '{cluster_name}' already exists. Updating spec if necessary...")
        cluster_api.patch_ray_cluster(
            name=cluster_name,
            ray_patch={"spec": cluster_spec["spec"]},
            k8s_namespace=tenant_namespace,
        )

    head_svc = f"{cluster_name}-head-svc.{tenant_namespace}.svc.cluster.local"
    ray_client_uri = f"ray://{head_svc}:10001"
    ray_dashboard_uri = f"http://{head_svc}:8265"

    print(f"⏳ Waiting for RayCluster '{cluster_name}' to become ready (timeout={timeout_seconds}s)...")
    is_ready = cluster_api.wait_until_ray_cluster_running(
        name=cluster_name,
        k8s_namespace=tenant_namespace,
        timeout=timeout_seconds,
        delay_between_attempts=5,
    )

    if not is_ready:
        status = cluster_api.get_ray_cluster_status(name=cluster_name, k8s_namespace=tenant_namespace)
        raise TimeoutError(f"RayCluster did not become ready within {timeout_seconds}s. Last status: {status}")

    print("\n✅ RayCluster is READY!")
    print(f"   Ray Client URI    : {ray_client_uri}")
    print(f"   Ray Dashboard URI : {ray_dashboard_uri}")
    return ray_client_uri, ray_dashboard_uri


def delete_ray_cluster(
    cluster_name: str = "raycluster-sample",
    tenant_namespace: str = TENANT_NAMESPACE,
) -> None:
    """Tear down a RayCluster using KubeRay SDK to release all compute resources."""
    cluster_api = kuberay_cluster_api.RayClusterApi()
    res = cluster_api.delete_ray_cluster(name=cluster_name, k8s_namespace=tenant_namespace)
    if res is not None:
        print(f"🗑️ Successfully deleted RayCluster '{cluster_name}' in tenant namespace '{tenant_namespace}'.")
    else:
        print(f"ℹ️ RayCluster '{cluster_name}' not found or already deleted.")

## Step 2: Provision the Ray Cluster & Wait for Readiness

Now we call `launch_ray_cluster()`. The KubeRay operator automatically creates the Head pod, Worker pods, and internal network endpoints:
- **Ray Client (`ray://...:10001`)**: Connect your interactive notebook session to execute tasks and actors.
- **Ray Dashboard (`http://...:8265`)**: Web UI for monitoring CPU/RAM utilization and submitting batch jobs.

In [ ]:
CLUSTER_NAME = "raycluster-sample"
NUM_WORKERS = 1

# Launch the cluster and wait until ready
RAY_CLIENT_URI, RAY_DASHBOARD_URI = launch_ray_cluster(
    cluster_name=CLUSTER_NAME,
    tenant_namespace=TENANT_NAMESPACE,
    num_workers=NUM_WORKERS,
    timeout_seconds=300,
)

## Step 2.1: Access the Ray Dashboard from Inside Your Workspace

You do **not** need `kubectl` or direct network access from your laptop! Because your Kubeflow Workspace pod runs directly inside the cluster, the Ray Head service (`:8265`) is reachable on the internal cluster network.

JupyterLab comes with **`jupyter-server-proxy`**, which forwards any port listening on `localhost` inside the workspace pod to your browser at `/proxy/<port>/`.

Below, we start a lightweight socket bridge inside the workspace pod to forward `127.0.0.1:8265` to the Ray Head service. You can then open the dashboard in a new tab or view it directly in the notebook!

In [ ]:
import os
import socket
import threading
from IPython.display import IFrame, display, HTML

# Track active Python forwarders so they can be cleanly stopped
_active_forwarders = {}

def stop_dashboard_port(local_port: int = 8265):
    """Stop an existing in-workspace port-forward."""
    if local_port in _active_forwarders:
        try:
            _active_forwarders[local_port].close()
        except Exception:
            pass
        del _active_forwarders[local_port]
        print(f"🛑 Closed port-forward on 127.0.0.1:{local_port}")
    else:
        print(f"ℹ️ No active Python forwarder tracked on port {local_port}.")

def forward_dashboard_port(cluster_name: str, tenant_namespace: str = TENANT_NAMESPACE, local_port: int = 8265):
    """Bridge local port in the workspace pod to the Ray head service in the cluster."""
    # Close any previously registered forwarder on this port
    if local_port in _active_forwarders:
        stop_dashboard_port(local_port)

    remote_host = f"{cluster_name}-head-svc.{tenant_namespace}.svc.cluster.local"
    server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    server.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
    try:
        server.bind(("127.0.0.1", local_port))
    except OSError as e:
        print(f"ℹ️ Port {local_port} is already active on localhost ({e}).")
        return
    server.listen(5)
    _active_forwarders[local_port] = server

    def handle_client(client_sock):
        remote_sock = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
        try:
            remote_sock.connect((remote_host, 8265))
        except Exception:
            client_sock.close()
            return
        def bridge(s1, s2):
            try:
                while True:
                    data = s1.recv(4096)
                    if not data: break
                    s2.sendall(data)
            except Exception:
                pass
            finally:
                s1.close()
                s2.close()
        threading.Thread(target=bridge, args=(client_sock, remote_sock), daemon=True).start()
        threading.Thread(target=bridge, args=(remote_sock, client_sock), daemon=True).start()

    def loop():
        while local_port in _active_forwarders:
            try:
                c, _ = server.accept()
                threading.Thread(target=handle_client, args=(c,), daemon=True).start()
            except OSError:
                break

    threading.Thread(target=loop, daemon=True).start()
    print(f"✅ Ray Dashboard port-forwarded inside workspace: 127.0.0.1:{local_port} -> {remote_host}:8265")

# Start forwarding the Ray Dashboard inside the workspace
forward_dashboard_port(CLUSTER_NAME, tenant_namespace=TENANT_NAMESPACE, local_port=8265)

# Render a link and embedded IFrame to view the dashboard directly in JupyterLab
nb_prefix = os.environ.get("NB_PREFIX", "/").rstrip("/") + "/"
dashboard_url = f"{nb_prefix}proxy/8265/"
display(HTML(f'🔗 Open Ray Dashboard in a new tab: <a href="{dashboard_url}" target="_blank"><b>{dashboard_url}</b></a>'))
display(IFrame(src=dashboard_url, width="100%", height=600))

## Step 3: Interactive Computing — Connect via Ray Client & Run Distributed Tasks

With your cluster ready, you can connect interactively using `ray.init(address=RAY_CLIENT_URI)`.

### How it works:
- Functions decorated with `@ray.remote` are distributed automatically across worker pods.
- Classes decorated with `@ray.remote` become distributed stateful services (**Actors**).
- Calling `.remote()` triggers asynchronous execution on a worker node and immediately returns an `ObjectRef` (future).
- `ray.get()` retrieves the return values from the worker nodes back into your notebook.

In [ ]:
import socket
import random
import ray

# 1. Connect to the remote Ray cluster
if ray.is_initialized():
    ray.shutdown()

ray.init(address=RAY_CLIENT_URI)

print("🔗 Connected to RayCluster!")
print("Cluster Resources:", ray.cluster_resources())

# 2. Define a distributed task with @ray.remote
@ray.remote
def estimate_pi_chunk(num_samples: int) -> tuple[int, str]:
    """Monte Carlo pi estimation running on a remote Ray worker node."""
    inside = 0
    for _ in range(num_samples):
        x, y = random.random(), random.random()
        if x * x + y * y <= 1.0:
            inside += 1
    # Return count and the hostname of the pod where this ran
    return inside, socket.gethostname()

# Submit parallel tasks across the worker nodes
NUM_TASKS = 8
SAMPLES_PER_TASK = 2_000_000
print(f"\n📊 Submitting {NUM_TASKS} tasks ({SAMPLES_PER_TASK:,} samples each) in parallel...")

futures = [estimate_pi_chunk.remote(SAMPLES_PER_TASK) for _ in range(NUM_TASKS)]
results = ray.get(futures)

total_inside = sum(count for count, _ in results)
worker_hosts = sorted({host for _, host in results})
pi_estimate = (4.0 * total_inside) / (NUM_TASKS * SAMPLES_PER_TASK)

print(f"🎯 Estimated Pi : {pi_estimate:.6f}")
print(f"🖥️ Tasks executed across {len(worker_hosts)} worker pods: {worker_hosts}")

# 3. Define a stateful Actor with @ray.remote
@ray.remote
class ParameterServer:
    """A distributed stateful Actor holding model parameters."""
    def __init__(self, dim: int):
        self.weights = [0.0] * dim

    def push_gradients(self, grads: list[float]) -> list[float]:
        self.weights = [w + g for w, g in zip(self.weights, grads)]
        return self.weights

    def get_weights(self) -> list[float]:
        return self.weights

ps = ParameterServer.remote(dim=4)
update_futures = [ps.push_gradients.remote([0.1 * i, 0.2 * i, 0.3 * i, 0.4 * i]) for i in range(1, 5)]
ray.get(update_futures)
print("\n🧠 Updated ParameterServer weights on worker:", ray.get(ps.get_weights.remote()))

# Disconnect interactive client session
ray.shutdown()
print("\n🔌 Disconnected interactive session.")

## Step 4: Submit Asynchronous Batch Jobs via Ray Jobs SDK

Interactive Ray Client (Step 3) is great for exploratory analysis. But for long-running training jobs or data pipelines, you should use the **Ray Jobs API** (`JobSubmissionClient`):

### Cluster Discovery & Selection:
In a team environment, multiple Ray clusters may exist in the namespace (for instance, an on-demand cluster like `raycluster-sample` that you spun up, and a shared team cluster like `raycluster-shared` deployed by administrators via `manifests/raycluster.yaml`).

Using KubeRay's `cluster_api.list_ray_clusters()`, you can discover all active clusters in your namespace, inspect their statuses, and choose which one to target with your batch job!

In [ ]:
import time
from ray.job_submission import JobSubmissionClient, JobStatus

# 1. Discover all active Ray clusters in the tenant namespace using KubeRay Python SDK
cluster_api = kuberay_cluster_api.RayClusterApi()
cluster_list = cluster_api.list_ray_clusters(k8s_namespace=TENANT_NAMESPACE)

print(f"📋 Discovering Ray Clusters in tenant namespace '{TENANT_NAMESPACE}':")
clusters_found = cluster_list.get("items", []) if cluster_list else []
for item in clusters_found:
    c_name = item["metadata"]["name"]
    c_state = item.get("status", {}).get("state", "pending")
    c_workers = item.get("status", {}).get("availableWorkerReplicas", 0)
    print(f"  • {c_name:<20} | Status: {c_state:<8} | Available Workers: {c_workers}")

# 2. Select target cluster: if a pre-provisioned 'raycluster-shared' exists, target it;
# otherwise use your interactive cluster CLUSTER_NAME
available_names = [c["metadata"]["name"] for c in clusters_found]
if "raycluster-shared" in available_names:
    target_cluster = "raycluster-shared"
    print(f"\n🎯 Selected shared batch cluster: '{target_cluster}'")
else:
    target_cluster = CLUSTER_NAME
    print(f"\n🎯 Selected interactive cluster: '{target_cluster}'")

target_dashboard_uri = f"http://{target_cluster}-head-svc.{TENANT_NAMESPACE}.svc.cluster.local:8265"

# 3. Package training script into a local directory
os.makedirs("ray_job_src", exist_ok=True)
with open("ray_job_src/train_job.py", "w") as f:
    f.write('''import ray
import socket
import time

# Initialize Ray within the job runtime on the cluster
ray.init()

@ray.remote
def distributed_step(epoch: int, shard_id: int):
    time.sleep(0.5)
    loss = 1.0 / (epoch + shard_id + 1)
    return {"epoch": epoch, "shard": shard_id, "loss": round(loss, 4), "pod": socket.gethostname()}

print("🚀 Starting distributed training job on Ray cluster...")
for epoch in range(1, 4):
    step_results = ray.get([distributed_step.remote(epoch, s) for s in range(4)])
    avg_loss = sum(r["loss"] for r in step_results) / len(step_results)
    pods = sorted({r["pod"] for r in step_results})
    print(f"Epoch {epoch} | Avg Loss: {avg_loss:.4f} | Worker Pods: {pods}")

print("✅ Training job completed successfully!")
''')

# 4. Submit the job using the Ray JobSubmissionClient SDK
job_client = JobSubmissionClient(target_dashboard_uri)

job_id = job_client.submit_job(
    entrypoint="python train_job.py",
    runtime_env={"working_dir": "./ray_job_src"},
)
print(f"📋 Submitted Ray Job ID: {job_id} to cluster '{target_cluster}'")

# 5. Stream status until completion
while True:
    status = job_client.get_job_status(job_id)
    print(f"   Current Job Status: {status}")
    if status in {JobStatus.SUCCEEDED, JobStatus.FAILED, JobStatus.STOPPED}:
        break
    time.sleep(3)

# 6. Fetch and display output logs
print("\n📜 --- Job Output Logs ---")
print(job_client.get_job_logs(job_id))

## Step 5: Tear Down Compute & Release Resources

When your distributed computations or batch jobs finish, shut down the Ray cluster using `delete_ray_cluster()`:
- Releases all worker and head pods back to the GKE pool, stopping compute billing.
- Your Kubeflow Notebook continues running safely, preserving all your files, code, and analysis outputs.

In [ ]:
# 1. Stop the local dashboard port-forward
stop_dashboard_port(local_port=8265)

# 2. Tear down the RayCluster via KubeRay SDK
delete_ray_cluster(cluster_name=CLUSTER_NAME, tenant_namespace=TENANT_NAMESPACE)